# Pricing GPU-Based Securities (GBS): Contract Options, Technology Jumps, and an ABS Benchmark
### Reproducible monthly Monte Carlo pricing, contract experiments, and numerical checks

Change **only the `DATA_DIR` line** in Section 1, with all original CSV and Excel files directly in that folder; then select **Kernel → Restart & Run All**. The notebook uses numpy, pandas, scipy and matplotlib with short, readable functions and no classes. A 60,000-path main calculation supplies every quoted baseline result; smaller samples are explicitly labeled supporting grids. The published contract terms are documented with source links; no additional input CSV files are needed.

## 1. Imports and local data directory

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.optimize import brentq

# >>> CHANGE ONLY THIS LINE: folder containing ALL supplied CSV and Excel files.
DATA_DIR = Path(r"C:/CHANGE_ME/GBS_Data")

OUTPUT_DIR = DATA_DIR / "GBS_JOD_results"
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"font.family":"DejaVu Sans", "figure.dpi":130,
                     "axes.spines.top":False, "axes.spines.right":False,
                     "axes.titlesize":12, "axes.labelsize":10,
                     "savefig.bbox":"tight"})
NAVY, CYAN, VIOLET, CORAL, GOLD = '#193653', '#1689AD', '#7558BA', '#E0744B', '#C29B3B'
print('Data directory:',DATA_DIR)

## 2. Observe GPU inputs, SOFR, and a term-structure proxy

Treasury **constant-maturity yields are not zero rates**. For a reproducible model benchmark, shift the Treasury CMT term-structure shape to the valuation-date SOFR one-month level and apply continuous discounting. This is a disclosed pricing proxy, not a traded OIS discount curve. The spread quotations are *different loan contracts* and are not GBS trade observations.

In [ ]:
specs = pd.read_csv(DATA_DIR/'01_gpu_specs.csv')
rental = pd.read_csv(DATA_DIR/'02_gpu_cloud_rental_prices.csv')
secondary = pd.read_csv(DATA_DIR/'03_gpu_secondary_market.csv')
fin = pd.read_csv(DATA_DIR/'04_gpu_backed_financing_transactions.csv')
adjacent = pd.read_csv(DATA_DIR/'06_adjacent_datacenter_abs.csv')
h15 = pd.read_csv(DATA_DIR/'FRB_H15.csv')
sofr = pd.read_excel(DATA_DIR/'SOFR.xlsx')
DATE = pd.Timestamp('2026-09-11')
sofr['Effective Date'] = pd.to_datetime(sofr['Effective Date'])
SOFR0 = float(sofr.loc[sofr['Effective Date'].eq(DATE),'Rate (%)'].iloc[0])/100
r = h15.loc[h15['Series Description'].eq(DATE.strftime('%Y-%m-%d'))].iloc[0]
tenors = np.array([1/12, 3/12, 6/12, 1, 2, 3, 5])
labels = ['1M','3M','6M','1Y','2Y','3Y','5Y']
keys = ['1-month','3-month','6-month','1-year','2-year','3-year','5-year']
treas = np.array([float(r[next(c for c in h15.columns if k in c)])/100 for k in keys])
term_curve = treas + (SOFR0-treas[0])
months = np.arange(1,49)
years = months/12
zero_proxy = np.interp(years,tenors,term_curve)
DF = np.exp(-zero_proxy*years)
guaranteed = rental[~rental['pricing_type'].str.contains('spot|preemptible',case=False,na=False)]
rent = guaranteed.groupby('gpu')['price_per_gpu_hour'].median().to_dict()
value = {}
for gpu in ('H100','H200'):
    v = secondary[(secondary.gpu==gpu) & (secondary.observation_type=='executed_median_90d')]
    value[gpu] = float(v.price_mid_usd.median())
v = secondary[(secondary.gpu=='B200') & (secondary.observation_type=='model_implied_value')]
value['B200'] = float(v.price_mid_usd.median())*.60
mix = {'H100':.50,'H200':.30,'B200':.20}
weighted_rent = sum(mix[g]*rent[g] for g in mix)
weighted_value = sum(mix[g]*value[g] for g in mix)
initial_util = .72
annual_gross_yield = weighted_rent*8760*initial_util/(weighted_value*1.30)
print('SOFR',round(SOFR0*100,3),'%; weighted price',round(weighted_rent,3),
      '$/GPU-hour; gross annualized service revenue / normalized collateral',round(annual_gross_yield,3))
print(pd.DataFrame({'tenor':labels,'treasury_CMT_pct':(treas*100).round(3),
                     'SOFR_shifted_pct':(term_curve*100).round(3)}).to_string(index=False))

## 3. Documented transaction terms: public contracts are not a new data file

Public records checked for the paper:
- CoreWeave (2026), SEC 8-K, DDTL 4.0: [SEC 8-K](https://www.sec.gov/Archives/edgar/data/1769628/000176962826000129/crwv-20260330.htm), [SEC 10-Q](https://www.sec.gov/Archives/edgar/data/1769628/000176962826000366/crwv-20260630.htm).
- IREN (2026), SEC 8-K, GPU financing: [SEC 8-K](https://www.sec.gov/Archives/edgar/data/1878848/000114036126023427/ef20075181_8k.htm), [2026 10-K](https://www.sec.gov/Archives/edgar/data/1878848/000187884826000052/iren-20260630.htm).
- Lambda (2026), $926m TLB: [issuer announcement](https://lambda.ai/blog/lambda-closes-926-million-senior-secured-term-loan-b-facility).
- KBRA (2026), Barings Equipment Finance 2026-A: [rating announcement](https://www.kbra.com/publications/wzxmHFgC/kbra-assigns-ratings-to-barings-equipment-finance-llc-2026-a?format=web).

Source-backed contract terms are **not** inferred from GPU quote snapshots. All numerical contract *experiment settings* below are clearly labeled illustrative.

In [ ]:
CONTRACT_FACTS = pd.DataFrame([
    ['CoreWeave DDTL 4.0',8500,'SOFR +225bp','2032-03-31','Delayed-draw; optional and mandatory prepayment; pledged GPU and customer agreement',
     'https://www.sec.gov/Archives/edgar/data/1769628/000176962826000129/crwv-20260330.htm'],
    ['IREN DDTL',1550,'1m SOFR +225bp','2031-12-31','Monthly scheduled amortization; shared security with fixed notes; DSCR covenants',
     'https://www.sec.gov/Archives/edgar/data/1878848/000114036126023427/ef20075181_8k.htm'],
    ['IREN notes',2100,'5.96% fixed','2031-12-31','Monthly amortization; common terms and security; project-linked customer contract',
     'https://www.sec.gov/Archives/edgar/data/1878848/000187884826000052/iren-20260630.htm'],
    ['Lambda TLB',926,'SOFR +300bp; OID 99.5','2030-12-31','Fully amortizing senior secured GPU infrastructure-backed SPV term loan',
     'https://lambda.ai/blog/lambda-closes-926-million-senior-secured-term-loan-b-facility'],
    ['Barings 2026-A equipment ABS',1030,'5.75% statistical discount rate','not reported here','Five note classes; leased/financed equipment; reserve and subordination; not GPU debt',
     'https://www.kbra.com/publications/wzxmHFgC/kbra-assigns-ratings-to-barings-equipment-finance-llc-2026-a?format=web'],
],columns=['transaction','amount_usdm','published_rate','reported_maturity','published_structure','public_source_url'])
display(CONTRACT_FACTS[['transaction','amount_usdm','published_rate','reported_maturity']])

## 4. Model settings (scenario inputs; not empirical estimates)

We normalize the physical asset base to 100. The **matched equipment-ABS counterfactual** keeps the same cash-flow mechanics, leverage, default hazard, and credit waterfall, but switches off the common GPU-generation shock. It is **not a valuation of Barings securities**.

In [ ]:
P = dict(T=48, price_drift=-.05, price_vol=.18, util_mean=.72,
         util_kappa=1.40, util_vol=.10, cost_share=.62, annual_dep=.22,
         tech_intensity=.15, price_retention=.82, value_retention=.72,
         customer_hazard=.05, anchor_share=.35, floor_share=0.0,
         redeploy_delay=3, redeploy_price=.84, redeploy_util=.84,
         redeploy_cost=.035, sale_fraction=.60, terminal_fraction=.80,
         reserve=2.0, senior=60.0, mezz=15.0, coupon_s=.0625,
         coupon_m=.10, dscr_trigger=0.0, cash_sweep_fraction=.80,
         call_hazard=0.0, call_tech_multiplier=.75, call_premium=0.0, technology_sweep=False)
# A contract experiment varies only one of these listed inputs at a time.
print(pd.Series(P).to_string())

## 5. Simulate the operating asset, joint jumps, default and recovery

The decision at customer failure compares contemporaneous forced-sale proceeds against **discounted future re-rental capacity net of downtime and an explicitly charged redeployment expense**. The continuation estimate is an underwriting approximation, not an ex-post look-ahead optimal stopping regression. The actual simulated cash flows pay the recovery cost in the failure month. Uniform random draws are reused between contractual counterfactuals.
The **generation-triggered sweep** begins in the shock month and continues until redemption; the notebook counts observed trigger months and actual dollars swept.


In [ ]:
def simulate_states(n=7000, seed=20260911, p=None, allow_redeployment=True):
    if p is None: p=P
    rng=np.random.default_rng(seed); T=int(p['T']); dt=1/12
    rent_index=np.ones(n); util=np.full(n,p['util_mean']); coll=np.full(n,100.)
    failed=np.zeros(n,bool); sold=np.zeros(n,bool); redeployed=np.zeros(n,bool)
    since=np.full(n,999,int); tech_seen=np.zeros(n,bool); refi_done=np.zeros(n,bool)
    cash=np.zeros((n,T)); call=np.zeros((n,T),bool); positive_income=np.zeros((n,T))
    default_events=np.zeros((n,T),bool); tech_events=np.zeros((n,T),bool); tech_history=np.zeros((n,T),bool)
    for t in range(T):
        z=rng.normal(size=n)
        rent_index *= np.exp((p['price_drift']-.5*p['price_vol']**2)*dt + p['price_vol']*np.sqrt(dt)*z)
        util += p['util_kappa']*(p['util_mean']-util)*dt + p['util_vol']*np.sqrt(np.maximum(util*(1-util),.01))*np.sqrt(dt)*rng.normal(size=n)
        util=np.clip(util,.20,.99)
        shock=rng.random(n)<(1-np.exp(-p['tech_intensity']*dt))
        tech_events[:,t]=shock; tech_seen |= shock; tech_history[:,t]=tech_seen
        rent_index[shock]*=p['price_retention']
        coll*=(1-p['annual_dep'])**dt; coll[shock]*=p['value_retention']
        newfail=(~failed)&(rng.random(n)<(1-np.exp(-p['customer_hazard']*dt)))
        default_events[:,t]=newfail; failed |= newfail
        since[failed]+=1; since[newfail]=0
        sale_pay=np.zeros(n); redep_bill=np.zeros(n)
        if newfail.any():
            idx=np.flatnonzero(newfail)
            sale_val=coll[idx]*p['anchor_share']*p['sale_fraction']
            rem=T-t-1
            now_ann=100*annual_gross_yield/12 * rent_index[idx]*(util[idx]/initial_util)*(1-p['cost_share'])*p['anchor_share']*p['redeploy_price']*p['redeploy_util']
            kk=np.arange(int(p['redeploy_delay'])+1,rem+1)
            annuity=np.sum(np.exp(-zero_proxy[min(t,47)]*kk/12)) if len(kk)>0 else 0.
            continuation=now_ann*annuity + p['anchor_share']*coll[idx]*(1-p['annual_dep'])**(rem/12)*p['terminal_fraction']*np.exp(-zero_proxy[min(t,47)]*rem/12)
            cost=100*p['redeploy_cost']*p['anchor_share']
            choose=(continuation-cost>sale_val)&allow_redeployment
            sold[idx[~choose]]=True; redeployed[idx[choose]]=True
            sale_pay[idx[~choose]]=sale_val[~choose]
            redep_bill[idx[choose]]=cost
        operating_factor=np.ones(n)
        operating_factor[failed & sold]=0
        operating_factor[failed & redeployed & (since<p['redeploy_delay'])]=0
        operating_factor[failed & redeployed & (since>=p['redeploy_delay'])]=p['redeploy_price']*p['redeploy_util']
        merchant=rent_index*(util/initial_util)
        protected=np.maximum(merchant,p['floor_share'])
        anchor_rent=np.where(failed,merchant*operating_factor,protected)
        income=100*annual_gross_yield/12*(1-p['cost_share'])*((1-p['anchor_share'])*merchant + p['anchor_share']*anchor_rent)
        positive_income[:,t]=income
        cash[:,t]=income + sale_pay - redep_bill
        # Callable refinancing is an EXTERNAL par principal repayment, not a sale of the same collateral.
        # Loan must be performing and have sufficient utilization; the refinancing decision is stylized.
        hazard=p['call_hazard']*(1+p['call_tech_multiplier']*shock.astype(float))
        newcall=(~refi_done)&(~failed)&(util>.55)&(rng.random(n)<(1-np.exp(-hazard*dt)))
        call[:,t]=newcall; refi_done|=newcall
    terminal=coll*p['terminal_fraction']*np.where(sold,1-p['anchor_share'],1.0)
    cash[:,-1]+=terminal
    return dict(cash=cash, income=positive_income, call=call,
                tech=tech_events, tech_history=tech_history, default=default_events, terminal=terminal,
                redeployed=redeployed, failed=failed)

## 6. Sequential monthly waterfall (interest arrears, reserves, principal and triggers)

Priority: senior interest (including arrears), senior scheduled principal (and arrears), mezzanine interest (including arrears), protection-triggered senior cash sweep, reserve replenishment, equity. At final maturity, senior principal and mezzanine principal are settled in order. When an **optional refinance call** occurs, *outside refinancing capital* repays outstanding debt at par plus the stated premium; the original credit securities terminate and future borrower revenue is not paid to those securities. This is different from collateral liquidation and must not be double-counted.

In [ ]:
def waterfall(states, c_s, c_m, p=None, n_s=None, n_m=None):
    if p is None: p=P
    a=states['cash']; n,T=a.shape
    S=float(p['senior'] if n_s is None else n_s); M=float(p['mezz'] if n_m is None else n_m)
    bal_s=np.full(n,S); bal_m=np.full(n,M); reserve=np.full(n,p['reserve'])
    i_s=np.zeros(n); i_m=np.zeros(n); active=np.ones(n,bool); unpaid_operating=np.zeros(n)
    cf_s=np.zeros((n,T)); cf_m=np.zeros((n,T)); cf_e=np.zeros((n,T)); princ_s=np.zeros((n,T))
    swept_by_path=np.zeros(n); trigger_months=0; binding_months=0
    for t in range(T):
        cash=a[:,t].copy()-unpaid_operating; cash[~active]=0
        # If redeployment costs exceed monthly inflow, unpaid operating costs are
        # carried to the following month instead of being erased or charged to equity.
        unpaid_operating=np.maximum(-cash,0)
        cash=np.maximum(cash,0)
        interest_s=(bal_s*c_s/12+i_s)*active
        scheduled=np.minimum(bal_s,np.maximum(0,S*(t+1)/T-(S-bal_s)))*active
        due_s=interest_s+scheduled
        draw=np.minimum(reserve,np.maximum(due_s-cash,0))*active
        cash+=draw; reserve-=draw
        interest_paid=np.minimum(cash,interest_s); cash-=interest_paid; i_s=interest_s-interest_paid
        principal_paid=np.minimum(cash,scheduled); cash-=principal_paid; bal_s-=principal_paid
        cf_s[:,t]+=interest_paid+principal_paid; princ_s[:,t]+=principal_paid
        interest_m=(bal_m*c_m/12+i_m)*active
        mezz_paid=np.minimum(cash,interest_m); cash-=mezz_paid; i_m=interest_m-mezz_paid
        cf_m[:,t]+=mezz_paid
        # A low debt-service-coverage observation traps discretionary cash for senior deleveraging.
        denom=np.maximum(interest_s+scheduled+interest_m,1e-7)
        coverage=states['income'][:,t]/denom
        trigger=(p['dscr_trigger']>0) & (coverage<p['dscr_trigger'])
        if p['technology_sweep']: trigger=trigger | states['tech_history'][:,t]
        trigger_months+=int(trigger.sum())
        swept=np.minimum(bal_s,cash*p['cash_sweep_fraction']*trigger)*active
        bal_s-=swept; cash-=swept; cf_s[:,t]+=swept; princ_s[:,t]+=swept
        swept_by_path+=swept; binding_months+=int((swept>1e-8).sum())
        # Contractual refinance call: external source pays principal and optional premium.
        call=states['call'][:,t]&active
        cf_s[call,t]+=bal_s[call]*(1+p['call_premium'])+i_s[call]
        cf_m[call,t]+=bal_m[call]*(1+p['call_premium'])+i_m[call]
        princ_s[call,t]+=bal_s[call]
        bal_s[call]=0; bal_m[call]=0; i_s[call]=0; i_m[call]=0
        cash[call]=0; reserve[call]=0; active[call]=False
        if t==T-1:
            cash+=reserve; reserve[:]=0
            xs=np.minimum(cash,i_s); cash-=xs; cf_s[:,t]+=xs; i_s-=xs
            xp=np.minimum(cash,bal_s); cash-=xp; cf_s[:,t]+=xp; princ_s[:,t]+=xp; bal_s-=xp
            xm=np.minimum(cash,i_m); cash-=xm; cf_m[:,t]+=xm; i_m-=xm
            mp=np.minimum(cash,bal_m); cash-=mp; cf_m[:,t]+=mp; bal_m-=mp
        else:
            replenish=np.minimum(np.maximum(p['reserve']-reserve,0),cash*.15)*active
            reserve+=replenish; cash-=replenish
        cf_e[:,t]+=np.maximum(cash,0)
    return dict(senior=cf_s,mezz=cf_m,equity=cf_e,
                senior_loss=bal_s/np.maximum(S,1e-9), mezz_loss=bal_m/np.maximum(M,1e-9),
                wal=np.mean(np.sum(princ_s*years,axis=1)/np.maximum(np.sum(princ_s,axis=1),1e-9)),
                sweep_mean=float(swept_by_path.mean()),trigger_months=trigger_months,
                binding_months=binding_months)

def pv_paths(cashflows):
    return cashflows @ DF

def tilt_weights(x, eta, scale):
    z=-eta*(x-x.mean())/max(scale,1e-6)
    z-=np.max(z)
    w=np.exp(np.clip(z,-65,0))
    return w/w.sum()

def quoted_price(cfs, w):
    return float(np.dot(w,pv_paths(cfs)))

def par_coupon(states,w,tranche='senior',p=None,n_s=None,n_m=None,coupon_s=None):
    if p is None:p=P
    notional=(p['senior'] if tranche=='senior' else p['mezz'])
    if tranche=='senior' and n_s is not None:notional=n_s
    if tranche=='mezz' and n_m is not None:notional=n_m
    sc=p['coupon_s'] if coupon_s is None else coupon_s
    def f(c):
        wf=(waterfall(states,c,p['coupon_m'],p,n_s,n_m) if tranche=='senior'
            else waterfall(states,sc,c,p,n_s,n_m))
        return quoted_price(wf[tranche],w)-notional
    c0,f0=0.,f(0.)
    if f0>=0:return 0.
    c1,f1=.12,f(.12)
    if f1<=0:
        return brentq(f,.12,.80,xtol=1e-5) if f(.8)>=0 else np.nan
    # Four short secant steps; f is continuous and monotone around the par coupon.
    for _ in range(4):
        c_new=np.clip(c1-f1*(c1-c0)/(f1-f0 if abs(f1-f0)>1e-10 else 1e-10),0,.8)
        f_new=f(c_new)
        if abs(f_new)<1e-4:return float(c_new)
        c0,f0,c1,f1=c1,f1,c_new,f_new
    return float(c1)

def fair_pair(states,w,p=None,n_s=None,n_m=None):
    if p is None:p=P
    cs=par_coupon(states,w,'senior',p,n_s,n_m)
    cm=par_coupon(states,w,'mezz',p,n_s,n_m,coupon_s=cs) if np.isfinite(cs) else np.nan
    return cs,cm

## 7. Price-state weights and observable spread anchor

The weights minimize sample relative entropy subject to a **single** benchmark moment. A fixed-rate equivalent annualized benchmark coupon is constructed from September 2026 SOFR + 225 bp (IREN's disclosed floating loan margin). The stylized 75-unit, 48-month straight-line loan is **not asserted to reproduce IREN's contractual amortization schedule**. It identifies one parameter, not an issuer-specific probability law or a unique no-arbitrage measure.
All headline valuation and external-quote results use **N = 60,000** identical main paths. A single calibration on the same main sample establishes the benchmark state-price tilt and its effective scenario count.


In [ ]:
N_MAIN=60000; MAIN_SEED=20260911
N=N_MAIN; BASE_SEED=MAIN_SEED
states=simulate_states(N,BASE_SEED,P,True)
BASE_STATES_SALE=simulate_states(N,BASE_SEED,P,False)
BENCH_SPREAD=225
bench_coupon=SOFR0+BENCH_SPREAD/10000
# A single-class senior secured benchmark, not the GBS senior class.
bench_p=P.copy(); bench_p['senior']=75.;bench_p['mezz']=0.;bench_p['reserve']=0.
bench_cash=waterfall(BASE_STATES_SALE,bench_coupon,0,bench_p,n_s=75,n_m=0)['senior']
bench_x=pv_paths(bench_cash); scale=float(bench_x.std())
def objective(e):return np.dot(tilt_weights(bench_x,e,scale),bench_x)-75.
# The moment may fail to identify a tilt if even the worst observable path repays at par.
grid=np.linspace(-30,80,111)
lohi=[(x,y) for x,y in zip(grid[:-1],grid[1:]) if objective(x)*objective(y)<=0]
if lohi:
    ETA=brentq(objective,*lohi[0],xtol=1e-9)
else:
    ETA=0.
    print('WARNING: target par moment outside simulated payoff support. Uniform weights retained.')
BASE_W=tilt_weights(bench_x,ETA,scale)
cs,cm=fair_pair(states,BASE_W)
base=waterfall(states,P['coupon_s'],P['coupon_m'])
print('Benchmark par calibration:', 'eta',round(ETA,4),'target',75,'fitted',round(objective(ETA)+75,4))
print('MAIN PATHS:',N_MAIN,'ESS:',round(1/(BASE_W@BASE_W)),'GBS par coupons:',cs,cm,'quoted contractual prices:',quoted_price(base['senior'],BASE_W),quoted_price(base['mezz'],BASE_W))

## 8. Matched ABS pricing and valuation helpers

The price measure uses a single base-case benchmark distribution, not a fresh risk-neutralization each time a contract changes. **All main exhibits use 60,000 paths with the same random seed and calibrated weights**. The equipment-ABS comparison retains the same legal waterfall, cash-flow level, and modeled contract defaults; it removes the common GPU-generation event. A separate case changes collateral only.


In [ ]:
# The main sample uses a single calibrated price measure and common random numbers.
def scenario(p, n=N_MAIN, seed=MAIN_SEED, stored_states=None, stored_weights=None):
    st=simulate_states(n,seed,p,True) if stored_states is None else stored_states
    if stored_weights is None:
        if n==N_MAIN and seed==MAIN_SEED:
            w=BASE_W
        else:
            # The reference process is held fixed even when contract terms change.
            ref=simulate_states(n,seed,P,False)
            loan=waterfall(ref,bench_coupon,0,bench_p,n_s=75,n_m=0)['senior']
            w=tilt_weights(pv_paths(loan),ETA,scale)
    else:
        w=stored_weights
    c1,c2=fair_pair(st,w,p)
    wf=waterfall(st,p['coupon_s'],p['coupon_m'],p)
    return {'price_s':quoted_price(wf['senior'],w),
            'price_m':quoted_price(wf['mezz'],w),
            'coupon_s':c1,'coupon_m':c2,
            'EL_s':float(np.dot(w,wf['senior_loss']))*100,
            'EL_m':float(np.dot(w,wf['mezz_loss']))*100,
            'WAL':wf['wal'],'redeploy':st['redeployed'].mean()*100,
            'sweep_mean':wf['sweep_mean'],'trigger_months':wf['trigger_months'],
            'binding_months':wf['binding_months']}

BASE=scenario(P,stored_states=states,stored_weights=BASE_W)
ABS_P=P.copy(); ABS_P.update(tech_intensity=0.,price_retention=1.,value_retention=1.)
ABS=scenario(ABS_P)
COLLATERAL_ONLY=P.copy(); COLLATERAL_ONLY['price_retention']=1.
COLL=scenario(COLLATERAL_ONLY)
for label,rr in [('Matched equipment ABS',ABS),('Collateral-only shock',COLL),('Joint GPU shock',BASE)]:
    print(label, '| senior',round(rr['price_s']/60*100,3),
          '| mezz',round(rr['price_m']/15*100,3),
          '| senior fair coupon %',round(rr['coupon_s']*100,3))


## 9. Exhibits 1–5: disclosed contracts, ABS comparison and pricing calibration

In [ ]:
def save_table(df,name):
    df.to_csv(OUTPUT_DIR/name,index=False)
    return df

save_table(CONTRACT_FACTS,'exhibit_01_contract_evidence.csv')
ex2=pd.DataFrame([['Equipment ABS (Barings 2026-A)','equipment loan/lease receivables','five note classes; OC, reserve, excess spread','ordinary servicing, prepayment, default, residual value'],
                  ['GPU-backed secured loan (CoreWeave)','GPU infrastructure + service contract','delayed draw; borrowing limits; repayments and hedging','technology; utilization; customer and collateral'],
                  ['GPU secured bank debt and notes (IREN)','GPU+Microsoft cashflow package','float DDTL + 5.96% fixed notes; shared collateral','DSCR and prepayment rights; operation continuity'],
                  ['Illustrative GBS contingent claim','pooled GPU service assets','senior / mezzanine / equity waterfall','joint jumps, default, refi, redeployment']],columns=['instrument','asset and contractual support','payment/credit structure','pricing state'])
save_table(ex2,'exhibit_02_product_comparison.csv')
ex3=pd.DataFrame({'GPU':['H100','H200','B200'], 'weight':[mix[x] for x in mix],
                  'rental_$/GPU_h':[rent[x] for x in mix], 'model_collateral_$/GPU':[value[x] for x in mix]})
save_table(ex3,'exhibit_03_gpu_calibration.csv')
ex4=pd.DataFrame({'term_years':tenors,'treasury_CMT_pct':treas*100,'SOFR_shifted_proxy_pct':term_curve*100})
save_table(ex4,'exhibit_04_interest_curve.csv')
ex5=pd.DataFrame([['SOFR date',str(DATE.date())],['SOFR %',SOFR0*100],['IREN floating margin (bp)',225],
                  ['Lambda announced OID',99.5],['Lambda announced floating margin bp',300],
                  ['Benchmark pricing tilt eta',ETA],['Effective state count',1/np.sum(BASE_W**2)],['Total main simulation paths',N_MAIN]],columns=['item','value'])
save_table(ex5,'exhibit_05_calibration_diagnostic.csv')
print('Exhibits 1-5 CSV saved')

## 10. Exhibit 6: observed quotations and a matched recovery-policy price bridge
The 75-unit IREN benchmark has **sale-only** recovery. The Lambda-like scenario was originally modeled with **redeployment rights**. These are different payoffs even when their face amount, coupon, discount factors and state-price tilt coincide. We price both conventions at +225bp and +300bp, and invert the 99.50 quote within each convention separately. All model values below use the same 60,000 simulated paths and state-price weights.

In [ ]:
# One hypothetical 75-unit loan, two explicitly different recovery payoffs.
# The sale-only benchmark is calibrated at +225bp; the redeployment version is a separate contract.
lambda_rate=SOFR0+.03
sale_cf=waterfall(BASE_STATES_SALE,lambda_rate,0,bench_p,n_s=75,n_m=0)['senior']
redeploy_cf=waterfall(states,lambda_rate,0,bench_p,n_s=75,n_m=0)['senior']
sale_x=pv_paths(sale_cf)
redeploy_x=pv_paths(redeploy_cf)
sale_300=quoted_price(sale_cf,BASE_W)/75*100
redeploy_300=quoted_price(redeploy_cf,BASE_W)/75*100
redeploy_225=quoted_price(waterfall(states,bench_coupon,0,bench_p,n_s=75,n_m=0)['senior'],BASE_W)/75*100

def loan_price_at_spread(spread_bp, loan_states):
    cfs=waterfall(loan_states,SOFR0+spread_bp/10000,0,bench_p,n_s=75,n_m=0)['senior']
    return quoted_price(cfs,BASE_W)/75*100

def loan_price_at_eta(e, discounted_cashflow):
    return float(tilt_weights(bench_x,e,scale)@discounted_cashflow)/75*100

implied_eta_sale=brentq(lambda e:loan_price_at_eta(e,sale_x)-99.5,-10.,25.)
implied_eta_redeploy=brentq(lambda e:loan_price_at_eta(e,redeploy_x)-99.5,-10.,25.)
implied_spread_sale=brentq(lambda bp:loan_price_at_spread(bp,BASE_STATES_SALE)-99.5,-300.,700.)
implied_spread_redeploy=brentq(lambda bp:loan_price_at_spread(bp,states)-99.5,-300.,700.)

ex6=pd.DataFrame([
    ['IREN reference (sale-only)',225.,100.,100.,'Par calibration: sale-only recovery'],
    ['Lambda TLB public quotation',300.,99.5,np.nan,'Public quotation; contract not fully replicated'],
    ['75-unit sale-only model',300.,np.nan,sale_300,'Same payoff as 225bp calibration'],
    ['75-unit redeployment model',225.,np.nan,redeploy_225,'Additional recovery right; same coupon as par anchor'],
    ['75-unit redeployment model',300.,np.nan,redeploy_300,'Additional recovery right; +300bp coupon'],
    ['Inverse tilt, redeploy',300.,99.5,99.5,f'eta = {implied_eta_redeploy:.4f}'],
    ['Inverse margin, redeploy',implied_spread_redeploy,99.5,99.5,'eta held at calibrated value'],
    ['Inverse margin, sale-only',implied_spread_sale,99.5,99.5,'eta held at calibrated value'],
    ['Illustrative GBS senior',np.nan,np.nan,BASE['price_s']/60*100,'60-unit tranche; coupon 6.25%'],
    ['Illustrative GBS mezzanine',np.nan,np.nan,BASE['price_m']/15*100,'15-unit tranche; coupon 10%']],
    columns=['comparison','quoted_or_implied_margin_bp','quoted_issue_pct','model_price_per100','interpretation'])
save_table(ex6,'exhibit_06_market_price_bridge.csv')
print('Sale-only: price(+225bp) = 100.00; price(+300bp) =',round(sale_300,3),
      '| 99.50-implied margin =',round(implied_spread_sale,2),'bp; implied eta =',round(implied_eta_sale,4))
print('Redeployment: price(+225bp) =',round(redeploy_225,3),
      '; price(+300bp) =',round(redeploy_300,3),
      '| 99.50-implied margin =',round(implied_spread_redeploy,2),
      'bp; implied eta =',round(implied_eta_redeploy,4))
assert abs(sale_300-101.301143)<.001 and abs(redeploy_300-102.989745)<.001
assert abs(implied_spread_sale-196.2143)<.05 and abs(implied_spread_redeploy-108.7476)<.05
display(ex6)

## 11. Exhibit 7: same security, conventional ABS versus GPU technology risk

In [ ]:
ex7=pd.DataFrame([['Matched equipment-ABS counterfactual',ABS],
                  ['Collateral-value shock only',COLL],
                  ['Coupled GPU rental/collateral jumps',BASE]],columns=['case','x'])
ex7=pd.concat([ex7['case'],ex7['x'].apply(pd.Series)],axis=1)
ex7['delta_senior_per100']=ex7.price_s/60*100-ABS['price_s']/60*100
ex7['delta_mezz_per100']=ex7.price_m/15*100-ABS['price_m']/15*100
save_table(ex7,'exhibit_07_abs_one_to_one.csv')
from matplotlib.ticker import MultipleLocator, FuncFormatter
# Two distinct x-axis scales keep small senior-price changes readable.
old_plot_style = plt.rcParams.copy()
labels = ['Matched equipment-ABS\ncounterfactual',
          'Collateral-value\nshock only',
          'Coupled GPU rental\nand collateral jumps']
d = ex7
out = OUTPUT_DIR/'exhibit_07_abs_comparison.png'
svg = OUTPUT_DIR/'exhibit_07_abs_comparison.svg'
s = d['delta_senior_per100'].to_numpy()
m = d['delta_mezz_per100'].to_numpy()
y = np.arange(3)
NAVY = '#15283f'
TEAL = '#0c8595'
PURPLE = '#7654b5'
GREY = '#637287'
LIGHT = '#e9edf3'

plt.rcParams.update({
    'font.family':'DejaVu Sans', 'font.size':11, 'axes.edgecolor':'#d9dfe8',
    'axes.labelcolor': NAVY, 'xtick.color':GREY, 'ytick.color':NAVY,
    'figure.facecolor':'white', 'axes.facecolor':'white', 'savefig.facecolor':'white'
})
fig = plt.figure(figsize=(13.2,6.0),dpi=180)
gs=fig.add_gridspec(1,2,left=.31,right=.965,top=.73,bottom=.245,wspace=.14,width_ratios=[1,1.15])
axs=[fig.add_subplot(gs[0,0]),fig.add_subplot(gs[0,1])]

# Title and concise subtitle outside plotting surfaces.
fig.text(.045,.943,'EXHIBIT 7',fontsize=11.6,weight='bold',color=TEAL,ha='left',va='top')
fig.text(.045,.897,'Incremental value of collateral and joint-compute shocks',fontsize=18.5,weight='bold',color=NAVY,ha='left',va='top')
fig.text(.045,.822,'Change in present value relative to the matched equipment-ABS counterfactual',fontsize=11.3,color=GREY,ha='left',va='top')

specs=[
  (axs[0],s,TEAL,'SENIOR TRANCHE',(-.40,.11),[-.4,-.3,-.2,-.1,0]),
  (axs[1],m,PURPLE,'MEZZANINE TRANCHE',(-10.7,1.5),[-10,-8,-6,-4,-2,0]),
]
for ax,values,color,panel_title,limits,ticks in specs:
    ax.set_xlim(*limits)
    ax.set_ylim(-.52,2.52)
    ax.invert_yaxis()
    ax.set_xticks(ticks)
    ax.tick_params(axis='x',labelsize=9.5,length=0,pad=9)
    ax.set_axisbelow(True)
    ax.grid(axis='x',color=LIGHT,linewidth=.9)
    ax.axvline(0,color='#95a1af',linewidth=1.2,zorder=1)
    ax.set_title(panel_title,loc='left',pad=18,fontsize=10.7,weight='bold',color=color)
    ax.set_xlabel('Price change per 100 of tranche face',labelpad=16,fontsize=10,color=GREY)
    ax.set_yticks(y)
    ax.tick_params(axis='y',length=0,pad=14)
    for spine in ax.spines.values(): spine.set_visible(False)
    for i,val in enumerate(values):
        if abs(val)<1e-12:
            ax.scatter(0,i,s=47,color=color,zorder=4,edgecolor='white',linewidth=.8)
        else:
            ax.barh(i,val,height=.29,color=color,alpha=.93,zorder=3)
            ax.scatter(val,i,s=32,color=color,edgecolor='white',linewidth=.8,zorder=4)
        if color == TEAL:
            txt = f'{val:+.3f}' if 0 < abs(val) < .01 else f'{val:+.2f}'
            if abs(val)<1e-12: txt='0.00'
            if val<-.06:
                xp=val-.015; ha='right'
            elif val<0:
                xp=val-.018;ha='right'
            else:
                xp=.012;ha='left'
        else:
            txt='0.00' if abs(val)<1e-12 else f'{val:+.2f}'
            xp=val-.20 if val < 0 else .22
            ha='right' if val<0 else 'left'
        ax.text(xp,i,txt,fontsize=10.8,weight='bold',color=color,ha=ha,va='center',zorder=5)

axs[0].set_yticklabels(labels,fontsize=10.5)
axs[1].set_yticklabels([])
# Draw small row rules spanning both panels on their own axes without obscuring bars.
for ax in axs:
    for yy in [.5,1.5]: ax.axhline(yy,color='#f1f3f6',linewidth=.9,zorder=0)

fig.text(.045,.084, 'Note: Values are model PV differences per 100 of tranche face; zero denotes the matched ABS case.',
         fontsize=9.4,style='italic',color=GREY,ha='left',va='center')
fig.text(.045,.047,'Panel axes use different scales so that senior-tranche changes remain visible.',
         fontsize=9.4,style='italic',color=GREY,ha='left',va='center')
fig.savefig(out,dpi=260,facecolor='white',bbox_inches=None)
fig.savefig(svg,format='svg',facecolor='white')
print(out,svg)

plt.show()
plt.rcParams.update(old_plot_style)


## 12. Exhibit 8: contractual floor, DSCR cash sweep and refinance call
The annual base refinancing intensity is 8%, with a generation-event multiplier parameter β_J = 0.75. A jump changes the intensity to 14% annualized *during the arrival month only*, conditional on a performing customer and utilization above 55%.

The price effects are **modeled contract sensitivities**, not contract terms inferred for particular real transactions. A floor of 0.70 means that the contracted anchor leg cannot fall below 70% of its initial service-level revenue while its customer remains solvent. A modeled 1.50 trigger redirects discretionary cash to senior debt (an illustrative threshold, not IREN’s published 1.05 maintenance or 1.10 prepayment condition). A refinancing call uses **external** principal capital and does not liquidate the GPU twice.
**Incremental figures use the same 60,000 paths and state-price weights, so the no-added-protection row is exactly the one reported in Exhibits 6 and 7.**


In [ ]:
variants=[('No added protection',{}),
          ('Take-or-pay floor 70%',{'floor_share':.70}),
          ('DSCR cash sweep 1.50x',{'dscr_trigger':1.50}),
          ('Technology-trigger sweep',{'technology_sweep':True}),
          ('Refinance call 8% hazard',{'call_hazard':.08}),
          ('Call 8% + 1% premium',{'call_hazard':.08,'call_premium':.01})]
rows=[]
for label,changes in variants:
    pp=P.copy();pp.update(changes)
    rr=BASE if not changes else scenario(pp)
    rows.append({'contract_case':label,**rr})
ex8=pd.DataFrame(rows)
ex8['delta_senior_per100']=(ex8.price_s-BASE['price_s'])/60*100
ex8['delta_mezz_per100']=(ex8.price_m-BASE['price_m'])/15*100
save_table(ex8,'exhibit_08_contractual_terms.csv')
fig,ax=plt.subplots(figsize=(10.0,4.5))
y=np.arange(len(ex8));h=.33
ax.barh(y-h/2,ex8.delta_senior_per100,height=h,label='Senior',color=CYAN)
ax.barh(y+h/2,ex8.delta_mezz_per100,height=h,label='Mezzanine',color=CORAL)
ax.set_yticks(y,ex8.contract_case);ax.invert_yaxis();ax.set_xlabel('Change in value per 100 of face vs no added protection')
ax.axvline(0,color=NAVY,linewidth=.9)
ax.set_title('Exhibit 8  |  Contract-contingent incremental valuation',loc='left',weight='bold')
ax.grid(axis='x',alpha=.15);ax.legend(frameon=False)
fig.tight_layout();fig.savefig(OUTPUT_DIR/'exhibit_08_contract_prices.png',dpi=240);plt.show()
print(ex8[['contract_case','delta_senior_per100','delta_mezz_per100','trigger_months','binding_months','sweep_mean']].to_string(index=False))


## 13. Exhibit 9: contract-implied par coupon and changes versus matched base

A call premium can lower the coupon required to issue at par. A negative **change in required coupon** is not a negative credit spread. This exhibit therefore reports coupon differences relative to the no-protection contract and reports weighted-average life separately.


In [ ]:
ex9=ex8[['contract_case','coupon_s','coupon_m','WAL','EL_s','EL_m','trigger_months','binding_months','sweep_mean']].copy()
ex9['senior_par_coupon_pct']=ex9.pop('coupon_s')*100
ex9['mezz_par_coupon_pct']=ex9.pop('coupon_m')*100
ex9['senior_delta_coupon_bp']=(ex9.senior_par_coupon_pct-ex9.senior_par_coupon_pct.iloc[0])*100
ex9['mezz_delta_coupon_bp']=(ex9.mezz_par_coupon_pct-ex9.mezz_par_coupon_pct.iloc[0])*100
save_table(ex9,'exhibit_09_contract_spreads_wal.csv')
display(ex9[['contract_case','senior_par_coupon_pct','mezz_par_coupon_pct',
             'senior_delta_coupon_bp','mezz_delta_coupon_bp','WAL']].round(3))


## 14. Exhibit 10: recovery choice and option value, properly net of conversion cost

In [ ]:
rows=[]
for sf in [.40,.55,.70,.85]:
    for delay in [1,3,6]:
        pp=P.copy();pp.update(sale_fraction=sf,redeploy_delay=delay)
        sale=simulate_states(2200,45454,pp,False)
        redep=simulate_states(2200,45454,pp,True)
        b=waterfall(sale,bench_coupon,0,bench_p,n_s=75,n_m=0)['senior']
        w=tilt_weights(pv_paths(b),ETA,scale)
        a=waterfall(sale,P['coupon_s'],P['coupon_m'],pp)
        c=waterfall(redep,P['coupon_s'],P['coupon_m'],pp)
        # Debt investors receive any gain in their particular contingent claims; total gain need not all reach debt.
        v_sale=quoted_price(a['senior']+a['mezz'],w)
        v_option=quoted_price(c['senior']+c['mezz'],w)
        rows.append(dict(sale_fraction=sf,delay=delay,debt_option_value=v_option-v_sale,
                         senior_option_value=quoted_price(c['senior']-a['senior'],w),
                         mezz_option_value=quoted_price(c['mezz']-a['mezz'],w)))
ex10=pd.DataFrame(rows);save_table(ex10,'exhibit_10_redeployment_option.csv')
mat=ex10.pivot(index='delay',columns='sale_fraction',values='debt_option_value')
fig,ax=plt.subplots(figsize=(7.5,4.25))
cmap=LinearSegmentedColormap.from_list('recovery',['#FFF3E4','#E7B273','#5D6FA6','#273D68'])
im=ax.imshow(mat.values,aspect='auto',cmap=cmap)
ax.set_xticks(np.arange(len(mat.columns)),[f'{v:.0%}' for v in mat.columns]);ax.set_yticks(np.arange(len(mat.index)),mat.index)
ax.set_xlabel('Forced-sale recovery / collateral value');ax.set_ylabel('Redeployment delay (months)')
for i in range(mat.shape[0]):
 for j in range(mat.shape[1]):
  val=mat.iloc[i,j];ax.text(j,i,f'{val:+.2f}',ha='center',va='center',color='white' if val>mat.values.max()*.55 else NAVY,fontweight='bold')
fig.colorbar(im,ax=ax,label='Incremental combined debt PV, normalized points')
ax.set_title('Exhibit 10  |  Net recovery flexibility and operating delay',loc='left',weight='bold')
fig.savefig(OUTPUT_DIR/'exhibit_10_recovery_heatmap.png',dpi=240);plt.show()

## 15. Exhibit 11: advance rate, credit enhancement and technology intensity

A reference **75% loan advance** is added alongside a 75-unit GBS senior-plus-15-unit mezzanine stack. These are different contractual claims: the single-class 75-unit sale-only loan used in the entropy calibration has **no mezzanine class and no reserve**, while the GBS series has both. The +225bp IREN margin is over SOFR rather than the matched fixed-coupon curve. The figure illustrates this difference explicitly rather than implying that the structures should price identically.


In [ ]:
rows=[]
GRID_N=12000;GRID_SEED=7712
ref=simulate_states(GRID_N,GRID_SEED,P,False)
loan=waterfall(ref,bench_coupon,0,bench_p,n_s=75,n_m=0)['senior']
GRID_W=tilt_weights(pv_paths(loan),ETA,scale)
for se in [50,55,60,65,70,75]:
    for tech in [0.,.15,.30]:
        pp=P.copy();pp.update(senior=se,tech_intensity=tech)
        rr=scenario(pp,n=GRID_N,seed=GRID_SEED,stored_weights=GRID_W)
        rows.append({'senior_size':se,'equity_size':100-se-15,'tech_intensity':tech,
                     'senior_margin_over_sofr_bp':(rr['coupon_s']-SOFR0)*10000,**rr})
ex11=pd.DataFrame(rows);save_table(ex11,'exhibit_11_subordination_technology.csv')
fig,axes=plt.subplots(1,2,figsize=(10.6,4.35),sharex=True)
for tech,color in [(0.,CYAN),(.15,VIOLET),(.30,CORAL)]:
    sub=ex11.loc[ex11.tech_intensity==tech].sort_values('senior_size')
    axes[0].plot(sub.senior_size,sub.senior_margin_over_sofr_bp,'o-',color=color,
                 label=f'Generation jump intensity: {tech:.0%}',linewidth=1.6)
    axes[1].plot(sub.senior_size,sub.price_m/15*100,'o-',color=color,linewidth=1.6)
axes[0].scatter([75],[225],marker='D',s=55,color=NAVY,zorder=7,
                label='75% single-class secured-loan anchor')
axes[0].set_ylim(50,245)
axes[0].set_title('Senior par coupon less spot SOFR (bp)',fontsize=10)
axes[1].set_title('Mezzanine value per 100 (10% coupon)',fontsize=10)
for ax in axes:
    ax.set_xlabel('Senior claim / eligible asset pool (%)',fontsize=9)
    ax.grid(alpha=.16)
    ax.axvline(75,linestyle='--',color='#9AA5B1',linewidth=.9)
fig.suptitle('Exhibit 11  |  Advance rate and technology exposure',fontsize=12,fontweight='bold',y=.98)
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,ncol=2,frameon=False,fontsize=8,
           loc='lower center',bbox_to_anchor=(.5,.005))
fig.tight_layout(rect=[0,.14,1,.89])
fig.savefig(OUTPUT_DIR/'exhibit_11_structure_jump.png',dpi=240)
plt.show()


## 16. Exhibit 12: 60,000-path numerical precision and rare-loss diagnostics

The table uses nested prefixes of the **same 60,000-path main random sample** and one fixed benchmark pricing parameter. For each sample, weights are normalized within that prefix; standard errors are calculated from the weighted self-normalized estimator. An increasing rare-loss point estimate is **not by itself convergence**. Report standard errors and the loss-event counts, and interpret narrow price intervals separately from tail-loss uncertainty.


In [ ]:
def weighted_mean_se(x,w):
    mean=float(w@x)
    # Delta-method self-normalized importance-sampling standard error.
    se=float(np.sqrt(len(x)/max(len(x)-1,1)*np.sum((w*(x-mean))**2)))
    return mean,se

audit=[]
main_loan=bench_x
main_note_s=pv_paths(base['senior'])/P['senior']*100
main_note_m=pv_paths(base['mezz'])/P['mezz']*100
loss_s=base['senior_loss']*100
for n in [15000,35000,N_MAIN]:
    w=tilt_weights(main_loan[:n],ETA,scale)
    ss,ss_se=weighted_mean_se(main_note_s[:n],w)
    mm,mm_se=weighted_mean_se(main_note_m[:n],w)
    ll,ll_se=weighted_mean_se(loss_s[:n],w)
    audit.append({'n_paths':n,'effective_n':1/np.sum(w**2),
                  'senior_price_per100':ss,'senior_price_SE':ss_se,
                  'mezz_price_per100':mm,'mezz_price_SE':mm_se,
                  'senior_EL_pct':ll,'senior_EL_SE_pct':ll_se,
                  'senior_loss_events':int(np.sum(loss_s[:n]>1e-8))})
ex12=pd.DataFrame(audit);save_table(ex12,'exhibit_12_convergence.csv')
print(ex12.round(4).to_string(index=False))
assert 0<SOFR0<.15
assert np.all(DF>0) and np.all(np.diff(DF)<0)
assert np.isfinite(BASE_W).all() and np.isclose(BASE_W.sum(),1)
assert np.isfinite(BASE['price_s']) and np.isfinite(BASE['price_m'])
assert np.isclose(BASE['price_s'],ex8.price_s.iloc[0])
assert np.isclose(BASE['price_m'],ex8.price_m.iloc[0])
assert np.isclose(BASE['price_s'],ex7.price_s.iloc[-1])
assert np.isclose(BASE['price_m'],ex7.price_m.iloc[-1])
assert np.isclose(ex12.senior_price_per100.iloc[-1],BASE['price_s']/60*100)
assert abs(objective(ETA))<.01
assert np.isfinite(implied_eta_sale) and np.isfinite(implied_eta_redeploy)
assert np.isfinite(implied_spread_sale) and np.isfinite(implied_spread_redeploy)
assert abs(loan_price_at_spread(implied_spread_sale,BASE_STATES_SALE)-99.50)<1e-6
assert abs(loan_price_at_spread(implied_spread_redeploy,states)-99.50)<1e-6
print('All exhibit numbers audited. Output folder:',OUTPUT_DIR)


## 17. Interpretation and replication

Tables and figures are written to `GBS_JOD_results/` inside the chosen input folder. The final benchmark, ABS comparison, and contract experiments use **60,000 identical main paths**; the supplementary credit-enhancement surface uses 12,000 paths per parameter pair with common random numbers. Exhibit 12 reports numerical standard errors and actual tail-event counts. The inverse Lambda comparison is explicitly a stylized shadow price, not a claim of reproducing Lambda's undisclosed detailed cash waterfall.
